# Small Models, Real Constraints

This notebook runs the whole project on Kaggle or Colab. Everything runs on a CPU, so no accelerator is needed.

**How to use it**

1. In the first code cell, choose `QUICK = True` or `QUICK = False`.
   - `QUICK = True` checks the full pipeline in a few minutes, using a 20,000-document training subsample and small sweeps. Its output goes to `results_quick/`.
   - `QUICK = False` reproduces the reported results. The main sweep takes about 42 minutes and the multi-seed runs about 68 more.
2. Run every cell from top to bottom.
3. The last cell zips the results and figures for download.

On Kaggle, turn **Internet** on in the side panel so the code and data can be downloaded.

In [ ]:
QUICK = True   # set to False for the full run (about two hours)

GITHUB_URL = "https://github.com/MosaabSaleem/Capstone"

import os, shutil, subprocess

# Work somewhere writable on either platform, and always start from a real folder.
if os.path.isdir("/kaggle/working"):
    BASE = "/kaggle/working"
elif os.path.isdir("/content"):
    BASE = "/content"
else:
    BASE = os.getcwd()
os.chdir(BASE)

target = os.path.join(BASE, "Capstone")
shutil.rmtree(target, ignore_errors=True)

result = subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, target],
                        capture_output=True, text=True)
if result.returncode != 0:
    print("git clone failed:", result.stderr)
    # Fall back to a copy of the repository attached as a Kaggle dataset.
    for root, dirs, files in os.walk("/kaggle/input"):
        if "requirements.txt" in files and "src" in dirs:
            shutil.copytree(root, target)
            print("Using the copy attached at", root)
            break

# Find the folder that actually holds the project, however it was nested.
project = next((d for d, dirs, files in os.walk(target)
                if "requirements.txt" in files and "src" in dirs), None)
assert project, "Could not find the project. Check GITHUB_URL or attach the repo as a dataset."
os.chdir(project)

FLAGS = "--quick" if QUICK else ""
RESULTS = "results_quick" if QUICK else "results"
FIGURES = "figures_quick" if QUICK else "figures"
print("Project folder:", os.getcwd())
print("Mode:", "quick check" if QUICK else "full run")

## 1. Install and record the environment

In [ ]:
!pip install -q -r requirements.txt
!python scripts/print_env.py {FLAGS}

## 2. Smoke test

Runs every stage on a tiny synthetic dataset. It should end with `SMOKE TEST PASSED`.

In [ ]:
!python scripts/smoke_test.py

## 3. Sparsity and baselines

In [ ]:
!python scripts/report_sparsity.py {FLAGS}
!python scripts/run_baselines.py {FLAGS}

## 4. Track A: pruning

Feature selection, document-frequency filtering and weight pruning. This is the slowest step, mostly because of mutual information.

In [ ]:
!python scripts/run_track_a.py {FLAGS}

## 5. Track B: quantisation

In [ ]:
!python scripts/run_track_b.py {FLAGS}

## 6. Rankings and the Pareto frontier

In [ ]:
!python scripts/analyse.py {FLAGS}

## 7. Multi-seed runs

Repeats the headline configurations under several seeds, then compares the feature selectors as paired observations.

In [ ]:
!python scripts/run_multiseed.py {FLAGS}
!python scripts/summarise_multiseed.py {FLAGS}

## 8. Figures

In [ ]:
!python scripts/make_figures.py {FLAGS}

import glob
from IPython.display import Image, display
for path in sorted(glob.glob(f"{FIGURES}/*.png")):
    print(os.path.basename(path))
    display(Image(filename=path))

## 9. Download the results

In [ ]:
import zipfile

archive = os.path.join(BASE, "capstone_outputs.zip")
with zipfile.ZipFile(archive, "w") as zf:
    for folder in (RESULTS, FIGURES):
        for path in glob.glob(f"{folder}/*"):
            zf.write(path)
print("Created", archive)

try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("On Kaggle, download capstone_outputs.zip from the Output panel.")